In [1]:
# Import required libraries
import json
import jsonlines
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from farasa.segmenter import FarasaSegmenter
from farasa.pos import FarasaPOSTagger
from farasa.ner import FarasaNamedEntityRecognizer
from tqdm.auto import tqdm
from collections import Counter


plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")

In [2]:
generated_by = 'openai'

In [3]:
# make sure the notebook is running from the project root
labeled_dataset_path = f'generated_arabic_datasets/{generated_by}/arabic_social_media_dataset/by_polishing_posts_generation_filtered.jsonl'

# Load human-AI text pairs
human_texts = []
ai_texts = []
with jsonlines.open(labeled_dataset_path) as reader:
    for obj in reader:
        if 'original_post' in obj and 'generated_post' in obj:
            # Clean and validate text
            human_text = obj['original_post'].strip()
            ai_text = obj['generated_post'].strip()
            human_texts.append(human_text)
            ai_texts.append(ai_text)

len(human_texts), len(ai_texts)

(3318, 3318)

In [4]:
def get_text_annotation(annotator, text):
    # annotator should be a function: e.g tagger.tag
    annotated_text = annotator(text).strip('S/S').strip('E/E').strip()
    processed_annotated_text = ''
    for word in annotated_text.split():
        if '/' not in word:
            processed_annotated_text += word + '/UN_TAGGED '
        else:
            processed_annotated_text += f'{word} '
    processed_annotated_text = processed_annotated_text.strip()
    words_tags = []
    for annotated_word in processed_annotated_text.split():
        split_annotated_word = annotated_word.split('/')
        words_tags.append((split_annotated_word[0], split_annotated_word[1]))
    return words_tags

In [5]:
human_texts[0]

"هنا الكثير من منيف، والكثير من شرق المتوسط أيضا. الأم هنا متعب الهذال في مدن الملح، أما أنيسة ورجب وحامد فهم الإنسان بكل تقلباته. لم أحب شخصية أنيسة، أزعجتني حقيقة. الرواية على أنها رائعة، وعلى أنها ذكرتني بالأشجار واغتيال مرزوق وجوها العام، إلا أنني توقعتها بشكل مختلف. توقعت السجون أرذل وأقذر وأقسى على الإنسان من هذا، ولا أقصد الأوصاف الحسية هنا. لعلي أرى ماتوقعت في 'الآن هنا..'. شرق المتوسط- عبدالرحمن منيف ???. سوف تنام طويلا، الموت والنوم متشابهان. لا فرق بينهما إلا ان الأول طويل والآخر قصير، ألا تنهض لنعيش فترة أطول؟ ص??. السجن والمرأة لا يجتمعان، وبداية انهيار السجين ان يسيطر عليه شبح امرأة. كفوا عن هذا المرض أيها الثيران، اخصوا أنفسكم لينتهي عذابكم! ص??. ... وانا من الذي ينتظرني؟ ص??. ... وفي تلك الدقائق، التي لم تكن تزيد على العشر أتزود بالقوة، بالجنون، بالمحبة، كنت أتزود منها لفترة طويلة تكفيني أسابيع، حتى عندما يمنعون الزيارة. ص??. وصمتت تاركة لدمعة كبيرة ان تسقط دون ان توقفها او تمسحها كما تعودت ان تفعل. ولما سألتها مرة أخرى، جاءت كلماتها غامضة حزينة:. - الكلب أمسكني من صدري

In [6]:
ai_texts[0]

'هنا الكثير من منيف، والكثير من شرق المتوسط أيضاً. الأم هنا متعب الهذال في مدن الملح، بينما أنيسة ورجب وحامد يمثلون الإنسان بكل تقلباته. لم أحب شخصية أنيسة، أزعجتني حقاً. الرواية على روعتها وعلى تذكيرها لي بالأشجار واغتيال مرزوق وجوّها العام، إلا أنني توقعتها مختلفة. توقعت السجون أن تكون أشرس وأقسى على الإنسان من ذلك، ولا أقصد الأوصاف الحسية هنا. ربما أرى ما توقعت في "الآن هنا...". شرق المتوسط - عبدالرحمن منيف... نوم طويل، الموت والنوم متشابهان. الفارق أنه الأول طويل والآخر قصير، ألا ننهض لنعش فترة أطول؟ ص??. اجتماع السجن والمرأة مستحيل، وانهيار السجين يبدأ بسيطرة طيف امرأة عليه. توقفوا عن هذا المرض أيها الثيران، اخصوا أنفسكم لتنهوا عذابكم! ص??... وأنا من الذي ينتظرني؟ ص??... وفي تلك الدقائق التي لم تكن تزيد عن العشر أستمد القوة، الجنون، المحبة منها لفترة طويلة تكفيني أسابيع، حتى عندما يمنعون الزيارة. ص??. وصمتت تاركة دمعة كبيرة تسقط دون أن توقفها أو تمسحها كما اعتادت. وعندما سألتها مرة أخرى، كانت كلماتها غامضة حزينة: "الكلب أمسكني من صدري"، وأشارت برأسها إلى الحارس الذي كان يدور حولنا

## POS Tagging

In [7]:
annotated_human_texts = []
annotated_ai_texts = []

In [8]:
pos_tagger = FarasaPOSTagger(interactive=True)

[2025-09-01 12:46:05,712 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [9]:
pos_tagger.tag(human_texts[0])

"S/S هنا/ADV ال+ كثير/DET+NOUN-MS من/PREP منيف/NOUN-MS ،/PUNC و+/CONJ ال+ كثير/DET+NOUN-MS من/PREP شرق/NOUN-MS ال+ متوسط/DET+NOUN-MS أيض/ADV +ا/CASE ./PUNC ال+ أم/DET+NOUN-MS هنا/NOUN-MS متعب/NOUN-MS ال+ هذال/DET+NOUN-MS في/PREP مدن/NOUN-FP ال+ ملح/DET+NOUN-MS ،/PUNC أما/PART أنيس +ة/NOUN+NSUFF-FD و+/CONJ رجب/NOUN-MS و+/CONJ حامد/NOUN-MS فهم/NOUN-MS ال+ إنسان/DET+NOUN-MS ب+/PREP كل/NOUN-MS تقلب +ات/NOUN+NSUFF-FP +ه/PRON ./PUNC لم/PART أحب/V شخصي +ة/NOUN+NSUFF-FS أنيس +ة/ADJ+NSUFF-FD ،/PUNC أزعجتني/NOUN-MS حقيق +ة/NOUN+NSUFF-FS ./PUNC ال+ رواي +ة/DET+NOUN+NSUFF-FS على/PREP أن/PART +ها/PRON رائع +ة/ADJ+NSUFF-FP ،/PUNC و+/CONJ على/PREP أن/PART +ها/PRON ذكرتني/NOUN-MS ب+/PREP ال+ أشجار/DET+NOUN-MP و+/CONJ اغتيال/NOUN-MS مرزوق/NOUN-MS و+/CONJ جو/NOUN-MS +ها/PRON ال+ عام/DET+ADJ-MS ،/PUNC إلا/PART أنني/V توقع +ت/V+PRON +ها/PRON ب+/PREP شكل/NOUN-MS مختلف/ADJ-MS ./PUNC توقع +ت/V+PRON ال+ سجون/DET+NOUN-MP أرذل/V و+/CONJ أقذر/NOUN-FP و+/CONJ أقسى/NOUN-MS على/PREP ال+ إنسان/DET+NOUN-MS من/PREP هذ

In [10]:
for human_text, ai_text in tqdm(zip(human_texts, ai_texts),total=len(human_texts)):
    annotated_human_texts.extend(get_text_annotation(pos_tagger.tag, human_text))
    annotated_ai_texts.extend(get_text_annotation(pos_tagger.tag, ai_text))

  0%|          | 0/3318 [00:00<?, ?it/s]

In [11]:
human_pos_counts = Counter([tag for word, tag in annotated_human_texts])
ai_pos_counts = Counter([tag for word, tag in annotated_ai_texts])
human_pos_counts, ai_pos_counts

(Counter({'UN_TAGGED': 1091235,
          'NOUN-MS': 631838,
          'PUNC': 489697,
          'PREP': 456733,
          'PRON': 398042,
          'V': 324118,
          'DET+NOUN-MS': 305911,
          'CONJ': 287660,
          'PART': 263363,
          'NOUN+NSUFF-FS': 102776,
          'DET+NOUN+NSUFF-FS': 92946,
          'ADJ-MS': 88188,
          'CASE': 75158,
          'V+PRON': 72599,
          'NOUN-FP': 57409,
          'DET+ADJ-MS': 57039,
          'DET+NOUN-MP': 38093,
          'NOUN+NSUFF-FD': 37126,
          'NUM-MP': 33582,
          'NOUN+NSUFF-FP': 33420,
          'DET+NOUN+NSUFF-FP': 25978,
          'DET+ADJ+NSUFF-FS': 22063,
          'ADV': 21463,
          'FOREIGN': 18563,
          'ADJ+NSUFF-FS': 18197,
          'NOUN-MP': 17196,
          'ADJ+NSUFF-FP': 14727,
          'DET+ADJ+NSUFF-FD': 13745,
          'DET+ADJ+NSUFF-FP': 13348,
          'ADJ+NSUFF-FD': 11895,
          'DET+NOUN+NSUFF-MP': 10880,
          'PREP+PART': 8937,
          'DET+NOUN+

In [12]:
# save counters as json files
with open(f'notebooks/stylometric_analysis/social_media_dataset/{generated_by}/human_pos_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        human_pos_counts, 
        f,
        ensure_ascii=False,
        indent=4,
    )
with open(f'notebooks/stylometric_analysis/social_media_dataset/{generated_by}/ai_pos_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        ai_pos_counts,
        f,
        ensure_ascii=False,
        indent=4,
    )

## NER

In [13]:
annotated_human_texts = []
annotated_ai_texts = []

In [14]:
ner_tagger = FarasaNamedEntityRecognizer(interactive=True)

[2025-09-01 12:56:04,806 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [15]:
ner_tagger.recognize(human_texts[0])

"هنا/O الكثير/O من/O منيف/B-PERS ،/O والكثير/O من/O شرق/O المتوسط/O أيضا/O ./O الأم/O هنا/O متعب/B-PERS الهذال/O في/O مدن/O الملح/O ،/O أما/O أنيسة/O ورجب/O وحامد/O فهم/O الإنسان/O بكل/O تقلباته/O ./O لم/O أحب/O شخصية/O أنيسة/O ،/O أزعجتني/O حقيقة/O ./O الرواية/O على/O أنها/O رائعة/O ،/O وعلى/O أنها/O ذكرتني/O بالأشجار/O واغتيال/O مرزوق/B-PERS وجوها/O العام/O ،/O إلا/O أنني/O توقعتها/O بشكل/O مختلف/O ./O توقعت/O السجون/O أرذل/O وأقذر/O وأقسى/O على/O الإنسان/O من/O هذا/O ،/O ولا/O أقصد/O الأوصاف/O الحسية/O هنا/O ./O لعلي/O أرى/O ماتوقعت/O في/O '/O الآن/O هنا/O ./O ./O '/O ./O شرق/B-LOC المتوسط/I-LOC -/O عبدالرحمن/B-PERS منيف/I-PERS ?/O ?/O ?/O ./O سوف/O تنام/O طويلا/O ،/O الموت/O والنوم/O متشابهان/O ./O لا/O فرق/O بينهما/O إلا/O ان/O الأول/O طويل/O والآخر/O قصير/O ،/O ألا/O تنهض/O لنعيش/O فترة/O أطول/O ؟/O ص/O ?/O ?/O ./O السجن/O والمرأة/O لا/O يجتمعان/O ،/O وبداية/O انهيار/O السجين/O ان/O يسيطر/O عليه/O شبح/O امرأة/O ./O كفوا/O عن/O هذا/O المرض/O أيها/O الثيران/O ،/O اخصوا/O أنفسكم/O ل

In [16]:
for human_text, ai_text in tqdm(zip(human_texts, ai_texts),total=len(human_texts)):
    annotated_human_texts.extend(get_text_annotation(ner_tagger.recognize, human_text))
    annotated_ai_texts.extend(get_text_annotation(ner_tagger.recognize, ai_text))

  0%|          | 0/3318 [00:00<?, ?it/s]

In [17]:
human_ner_counts = Counter([tag for word, tag in annotated_human_texts])
ai_ner_counts = Counter([tag for word, tag in annotated_ai_texts])
human_ner_counts, ai_ner_counts

(Counter({'O': 3271872,
          'B-PERS': 32309,
          'I-PERS': 19899,
          'B-LOC': 15986,
          'B-ORG': 3974,
          'I-ORG': 2016,
          '': 1916,
          'I-LOC': 1181,
          'I-PER': 58,
          '10': 57,
          'B-PER': 51,
          '5': 43,
          '.': 37,
          '2': 8,
          '4': 7,
          '3': 6,
          '7': 5,
          '0': 5,
          '6': 4,
          '1': 4,
          '04': 3,
          '01': 3,
          '12': 3,
          '06': 3,
          '90': 3,
          '02': 2,
          '08': 2,
          '05': 2,
          '553': 1,
          '265': 1,
          '186': 1,
          '142': 1,
          '228': 1,
          '23': 1,
          '11': 1,
          '4552': 1,
          '26': 1,
          '72129': 1,
          '03': 1,
          '09': 1,
          '170': 1,
          '2003': 1,
          '24': 1,
          '72': 1,
          '54': 1,
          '230': 1,
          '185': 1,
          '951': 1,
          '231': 1,
   

In [18]:
# save counters as json files
with open(f'notebooks/stylometric_analysis/social_media_dataset/{generated_by}/human_ner_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        human_ner_counts, 
        f,
        ensure_ascii=False,
        indent=4,
    )
with open(f'notebooks/stylometric_analysis/social_media_dataset/{generated_by}/ai_ner_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        ai_ner_counts,
        f,
        ensure_ascii=False,
        indent=4,
    )

## Segmentation

In [19]:
annotated_human_texts = []
annotated_ai_texts = []

In [20]:
segmenter = FarasaSegmenter(interactive=True)

[2025-09-01 15:07:43,591 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [21]:
segmenter.segment(human_texts[0])

"هنا ال+كثير من منيف ، و+ال+كثير من شرق ال+متوسط أيض+ا . ال+أم هنا متعب ال+هذال في مدن ال+ملح ، أما أنيس+ة و+رجب و+حامد فهم ال+إنسان ب+كل تقلب+ات+ه . لم أحب شخصي+ة أنيس+ة ، أزعجتني حقيق+ة . ال+رواي+ة على أن+ها رائع+ة ، و+على أن+ها ذكرتني ب+ال+أشجار و+اغتيال مرزوق و+جو+ها ال+عام ، إلا أنني توقع+ت+ها ب+شكل مختلف . توقع+ت ال+سجون أرذل و+أقذر و+أقسى على ال+إنسان من هذا ، و+لا أقصد ال+أوصاف ال+حسي+ة هنا . ل+علي أرى ماتوقعت في ' الآن هنا . . ' . شرق ال+متوسط - عبدالرحمن منيف ? ? ? . سوف تنام طويل+ا ، ال+موت و+ال+نوم متشابه+ان . لا فرق بين+هما إلا ان ال+أول طويل و+ال+آخر قصير ، ألا تنهض ل+نعيش فتر+ة أطول ؟ ص ? ? . ال+سجن و+ال+مرأ+ة لا يجتمعان ، و+بداي+ة انهيار ال+سجين ان يسيطر علي+ه شبح امرأ+ة . كف+وا عن هذا ال+مرض أيها ال+ثيران ، اخص+وا أنفس+كم ل+ينتهي عذاب+كم ! ص ? ? . . .. و+أنا من الذي ينتظرني ؟ ص ? ? . . .. و+في تلك ال+دقائق ، التي لم تكن تزيد على ال+عشر أتزود ب+ال+قو+ة ، ب+ال+جنون ، ب+ال+محب+ة ، كن+ت أتزود من+ها ل+فتر+ة طويل+ة تكفيني أسابيع ، حتى عندما يمنع+ون ال+زيار+ة . ص ? ? . و+صمت+

In [22]:
def process_segmentations(annotated_text):
    all_subtokens = list()
    for token in annotated_text.split():
        subtokens = token.split('+')
        subtokens = [subtoken.strip() for subtoken in subtokens]  # noqa: E731
        all_subtokens.extend(subtokens)
    return all_subtokens

In [23]:
human_segmentations = []
ai_segmentations = []

In [24]:
for human_text, ai_text in tqdm(zip(human_texts, ai_texts),total=len(human_texts)):
    human_segmentations.extend(process_segmentations(segmenter.segment(human_text)))
    ai_segmentations.extend(process_segmentations(segmenter.segment(ai_text)))

  0%|          | 0/3318 [00:00<?, ?it/s]

In [25]:
human_segmentations[:10], ai_segmentations[:10]

(['هنا', 'ال', 'كثير', 'من', 'منيف', '،', 'و', 'ال', 'كثير', 'من'],
 ['هنا', 'ال', 'كثير', 'من', 'منيف', '،', 'و', 'ال', 'كثير', 'من'])

In [26]:
human_segmentations_counts = Counter(human_segmentations)
ai_segmentations_counts = Counter(ai_segmentations)
human_segmentations_counts.most_common(5), ai_segmentations_counts.most_common(5)

([('ال', 595812), ('ة', 314562), ('و', 237583), ('.', 218785), ('ه', 134929)],
 [('ال', 346664), ('ة', 193906), ('و', 113653), ('،', 98991), ('.', 94890)])

In [27]:
# save counters as json files
with open(f'notebooks/stylometric_analysis/social_media_dataset/{generated_by}/human_segmentations_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        human_segmentations_counts, 
        f,
        ensure_ascii=False,
        indent=4,
    )
with open(f'notebooks/stylometric_analysis/social_media_dataset/{generated_by}/ai_segmentations_counts.json', 'w', encoding='utf-8') as f:
    json.dump(
        ai_segmentations_counts,
        f,
        ensure_ascii=False,
        indent=4,
    )